# 11. 최종 보행 열스트레스 지수 (⑤ 재구성) — deliverable 1

PHSI(05)·UTCI(07) 이원화 해소 → **단일 지수**. 균일한 UTCI 대신 **판별력 강한 02 표면조성**을 축으로.

`HSI = norm( 0.5·물리열(LST) + 0.5·불투수 − 0.5·냉각(0.6산림+0.4수역) )`

- 물리열: lst_peak(관측 피크). 불투수: 뜨겁고 그늘없는 표면. 냉각: 산림그늘+수역증발.
- UTCI(07)는 **시간대별 절대 심각도**(언제 위험) 동반지표. 이 HSI는 **공간 우선순위**(어디 위험).

산출물: `data/results/heat_index_final.geojson`, `output/figures/fig14_hsi.png`

In [1]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd, geopandas as gpd
from src.config import DATA_PROCESSED, DATA_RESULTS, OUT_FIGURES, CRS_SUBMIT

g = gpd.read_file(DATA_PROCESSED / "grid_features.geojson")

def norm(s):
    lo, hi = s.quantile(0.02), s.quantile(0.98)
    return ((s - lo) / (hi - lo)).clip(0, 1)

g["z_heat"] = norm(g["lst_peak"])
g["z_cool"] = 0.6 * g["tree_frac"] + 0.4 * g["water_frac"]
raw = 0.5 * g["z_heat"] + 0.5 * g["imperv_frac"] - 0.5 * g["z_cool"]
g["hsi"] = norm(raw)
g["hsi_level"] = pd.qcut(g["hsi"], 5, labels=["매우낮음", "낮음", "보통", "높음", "매우높음"]).astype(str)   # Categorical -> str (gpd 0.13 호환)
print("HSI 지구별 평균:")
print(g.groupby("zoneName")["hsi"].mean().round(3).sort_values(ascending=False))
print("HSI 분포:", {k: round(float(v), 2) for k, v in g["hsi"].describe()[["mean", "min", "max", "std"]].items()})

HSI 지구별 평균:
zoneName
성남분당             0.667
성남판교지구 택지개발사업    0.604
남양주왕숙 공공주택지구     0.437
고양창릉 공공주택지구      0.220
Name: hsi, dtype: float64
HSI 분포: {'mean': 0.52, 'min': 0.0, 'max': 1.0, 'std': 0.29}


In [2]:
# 판별력 비교: HSI vs (구)UTCI 주간평균 — 지구 표준편차
u = gpd.read_file(DATA_RESULTS / "heat_index_utci.geojson")[["grid_id", "utci_daymean"]]
g2 = g.merge(u, on="grid_id")
print("공간 판별력(격자 표준편차, 0~1 정규화 기준):")
print("  HSI std:", round(float(g["hsi"].std()), 3), "(범위", round(float(g.hsi.max()-g.hsi.min()),2), ")")
print("  UTCI std:", round(float(norm(g2["utci_daymean"]).std()), 3))
print("\n최상위 취약 2000격자 성분평균:")
top = g.nlargest(2000, "hsi")
print({c: round(float(top[c].mean()), 2) for c in ["z_heat", "imperv_frac", "tree_frac", "water_frac"]})

공간 판별력(격자 표준편차, 0~1 정규화 기준):
  HSI std: 0.292 (범위 1.0 )
  UTCI std: 0.279

최상위 취약 2000격자 성분평균:
{'z_heat': 0.95, 'imperv_frac': 0.98, 'tree_frac': 0.0, 'water_frac': 0.0}


In [3]:
cols = ["grid_id", "zoneName", "blockType", "z_heat", "imperv_frac", "tree_frac", "water_frac",
        "hsi", "hsi_level", "peak_lowconf", "geometry"]
g[cols].to_crs(CRS_SUBMIT).to_file(DATA_RESULTS / "heat_index_final.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

import matplotlib.pyplot as plt
from src.viz.fonts import setup_font
setup_font()   # 한글 폰트 자동 적용(COMPAS 미보유 시 동봉 폰트 등록)
fig, ax = plt.subplots(1, 2, figsize=(15, 8))
g.plot(column="hsi", cmap="YlOrRd", ax=ax[0], linewidth=0, legend=True, legend_kwds={"shrink": 0.6})
ax[0].set_title("보행 열스트레스 지수 HSI (표면기반)")
g.plot(column="hsi_level", cmap="YlOrRd", ax=ax[1], linewidth=0, legend=True, categorical=True)
ax[1].set_title("HSI 5등급")
for a in ax: a.set_axis_off()
plt.tight_layout(); plt.savefig(OUT_FIGURES / "fig14_hsi.png", dpi=130); plt.close()